# Kelmarsh energy forecast: Mamba2 on a Kaggle GPU

Trains the framework's Mamba2 on `Energy Export (kWh)` per 10-minute slot, six turbines in one
model, 36 slots of history, a 10-slot horizon, test period 2023-2024.

**Before running:** Settings, Accelerator: one GPU (P100 or T4), Internet: on (for `pip`).
Add the dataset holding `kelmarsh-mamba2-bundle.zip` (Kaggle unzips it under `/kaggle/input/`).

In [ ]:
import subprocess, sys, torch
print(sys.version)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")

In [ ]:
# Only what Kaggle lacks. torch, numpy and scikit-learn stay as Kaggle ships them.
!pip install -q polars==1.44.2 python-dotenv mlflow

In [ ]:
import glob, os
script = glob.glob("/kaggle/input/**/kelmarsh_energy_forecast.py", recursive=True)
assert script, "bundle not found: add the dataset with kelmarsh-mamba2-bundle.zip to this notebook"
BUNDLE = os.path.dirname(script[0])
os.environ.update(PYTHONPATH=BUNDLE, MLFLOW_ENABLED="false")
OUT = "/kaggle/working/report"
print(BUNDLE, os.listdir(BUNDLE))

In [ ]:
# Import check: a failure here is a missing or too-new dependency, not the model.
!cd /kaggle/working && python -c "import ml_framework.base_models.configs as c; print(c.BaseModelsEnum.MAMBA2_REGRESSOR)"

## 1. Quick check (a few minutes)
Two epochs on a small slice, on the GPU. If this fails, stop here.

In [ ]:
!cd /kaggle/working && python $BUNDLE/kelmarsh_energy_forecast.py --model mamba2 --device cuda \
    --data $BUNDLE/kelmarsh_10min.parquet --start 2023-06-01 --cutoff 2024-06-01 --end 2024-07-01 \
    --param epochs=2 --report /kaggle/working/check

## 2. Timing: one epoch over all the training years
The summary's `seconds_total` holds the framing, one epoch and the prediction over a one-month test.
The prediction runs on the CPU (the model's design) and is the part that grows with the test period.

In [ ]:
!cd /kaggle/working && python $BUNDLE/kelmarsh_energy_forecast.py --model mamba2 --device cuda \
    --data $BUNDLE/kelmarsh_10min.parquet --start 2016-01-01 --cutoff 2022-12-01 --end 2023-01-01 \
    --param epochs=1 --report /kaggle/working/timing

## 3. The full run
Train 2016-2022, test 2023-2024, the model's own epochs (up to 200, early stopping after 10).

In [ ]:
!cd /kaggle/working && python $BUNDLE/kelmarsh_energy_forecast.py --model mamba2 --device cuda \
    --data $BUNDLE/kelmarsh_10min.parquet --start 2016-01-01 --cutoff 2023-01-01 --end 2025-01-01 \
    --report $OUT

In [ ]:
import json, glob
for path in glob.glob(OUT + "/summary_*.json"):
    r = json.load(open(path)); m = r["metrics"]
    print(path.split("/")[-1], "| epochs", r["epochs_run"], "best", r["best_epoch"],
          "| MAE %.2f kWh (persistence %.2f) skill %+.3f R2 %.4f | %.0f s"
          % (m["MAE_mean"], m["persistence_MAE_mean"], m["skill_MAE_mean"], m["R2_mean"], r["seconds_total"]))